In [40]:
import cirq
#Syntactic Check -- Makes sure that everything actually runs 
def preformat_code(raw_code: str) -> dict:
 
 if raw_code.startswith("```python"):
    llm_code = raw_code.removeprefix("```python").removesuffix("```").strip()
 elif raw_code.startswith("python\n"):
    llm_code = raw_code.removeprefix("python\n")
 else:
    llm_code = raw_code or ""
 return {"llm_code": llm_code}

def check_syntax(llm_code: str) -> dict:

    try:
        exec_globals = {}
        exec(llm_code, exec_globals)
        return {"passed": True, "error": None}
    except Exception as e:
        return {"passed": False, "error": str(e)}

In [39]:
#Structural Check -- does it have the right gates and qubits
def check_structure(llm_circuit: cirq.Circuit, control_circuit: cirq.Circuit, original_prompt: str = "", llm_code: str = "") -> dict:
    results = {}

    # Check amount of qubits
    llm_qubits = len(llm_circuit.all_qubits())
    control_qubits = len(control_circuit.all_qubits())
    results["qubit_count_match"] = llm_qubits == control_qubits

    # Check gate type present
    def get_gates(circuit):
        return {
            type(op.gate).__name__
            for moment in circuit
            for op in moment.operations
            if getattr(op, "gate", None) is not None
    }

    llm_gates = get_gates(llm_circuit)
    control_gates = get_gates(control_circuit)
    results["gate_types_match"] = control_gates.issubset(llm_gates)
    results["llm_gates"] = sorted(llm_gates)
    results["control_gates"] = sorted(control_gates)

    # Measurement Check

    llm_has_measurement = any(

        isinstance(op.gate, cirq.MeasurementGate)

        for moment in llm_circuit
        for op in moment.operations
    )

    control_has_measurement = any(
        isinstance(op.gate, cirq.MeasurementGate)
        for moment in control_circuit
        for op in moment.operations
    )
    results["measurement_match"] = llm_has_measurement == control_has_measurement

    # Simulation Check
    simulation_keywords = [
        "cirq.simulator",
        "simulator(",
        ".run(",
        ".simulate(",
        "run(",
        "simulate(",
        "sample(",
        "aer",
        "backend.run",
        "qiskit",
    ]

    def check_for_simulation(code_text: str) -> bool:
        code_lower = (code_text or "").lower()
        return any(keyword in code_lower for keyword in simulation_keywords)

    def validate_simulation_match(prompt: str, llm_code: str) -> dict:
        prompt_lower = (prompt or "").lower()
        prompt_asks_for_simulation = any(
            keyword in prompt_lower
            for keyword in [
                "simulate",
                "simulation",
                "run the circuit",
                "run circuit",
                "results",
                "execute",
                "sample",
                "measurement results",
                "output",
            ]
        )
        code_has_simulation = check_for_simulation(llm_code)
        return {
            "simulation_requested": prompt_asks_for_simulation,
            "code_contains_simulation": code_has_simulation,
            "simulation_match": (
                (prompt_asks_for_simulation and code_has_simulation)
                or (not prompt_asks_for_simulation and not code_has_simulation)
            ),
        }

    simulation_result = validate_simulation_match(original_prompt, llm_code)
    results["simulation_requested"] = simulation_result["simulation_requested"]
    results["code_contains_simulation"] = simulation_result["code_contains_simulation"]
    results["simulation_match"] = simulation_result["simulation_match"]

    results["circuit_depth_llm"] = len(llm_circuit)
    results["circuit_depth_control"] = len(control_circuit)
    results["depth_within_tolerance"] = len(llm_circuit) <= len(control_circuit) * 2

    return results

In [41]:
# Semantic Check - checks the output distribution
import numpy as np

def check_semantics(llm_circuit: cirq.Circuit, control_circuit: cirq.Circuit, reps: int = 1000, tolerance: float = 0.1) -> dict:
    simulator = cirq.Simulator()

#Measurement Ensured

    def ensure_measured(circuit):
        qubits = sorted(circuit.all_qubits())
        if not any(
            isinstance(op.gate, cirq.MeasurementGate)
            for moment in circuit
            for op in moment.operations
        ):
            circuit = circuit + cirq.measure(*qubits, key='result')
        return circuit

    llm_measured = ensure_measured(llm_circuit.copy())
    control_measured = ensure_measured(control_circuit.copy())

    llm_result = simulator.run(llm_measured, repetitions=reps)
    control_result = simulator.run(control_measured, repetitions=reps)

    #Build Probability(Result) distributions

    def get_distribution(result, reps):
        counts = result.multi_measurement_histogram(keys=['result']) # counts the number of outcomes 
        return {k: v/ reps for k, v in counts.items()}  #turns number into probability
    llm_dist = get_distribution(llm_result, reps)
    control_dist = get_distribution(control_result, reps)

    #Compare distributions 

    all_keys = set(llm_dist.keys()) | set(control_dist.keys())

    distribution_match = all(
        abs(llm_dist.get(k, 0) - control_dist.get(k, 0)) <= tolerance
        for k in all_keys
    )

    # STATE VECTOR
    def get_state_vector(circuit):
        no_measurement = cirq.drop_terminal_measurements(circuit)
        return cirq.final_state_vector(no_measurement)

    llm_vector = get_state_vector(llm_circuit)
    control_vector = get_state_vector(control_circuit)

    #comparing the vectors
    fidelity = float(abs(np.dot(np.conj(llm_vector), control_vector))**2)
    assert cirq.equal_up_to_global_phase(llm_vector, control_vector)
    # np.dot(np.conj()creates inner products of the two vectors 
    # the absolute value and square on it makes the quantum fidelity 
    # quantum fidelity of 1 means that the two vectors are the same and 0 means that they are orthagonal or completely different. 
    
    # RETURN combined result
    return {
        "passed": distribution_match and fidelity >= 0.9,
        "fidelity_score": fidelity,
        "distribution_match": distribution_match,
        "llm_distribution": {str(k): v for k, v in llm_dist.items()},
        "control_distribution": {str(k): v for k, v in control_dist.items()},
    }

In [42]:
# THE BIG ONE

def validate_circuit(llm_code: str, control_circuit: cirq.Circuit, original_prompt: str) -> dict:
    report = {}

    # Step 1: Syntax
    syntax = check_syntax(llm_code)
    report["syntax"] = syntax

    if not syntax["passed"]:
        report["verdict"] = "invalid_code"
        return report

    # Step 2: Extracting da circuit
    exec_globals = {}
    exec(llm_code, exec_globals)
    llm_circuit = next(
        (v for v in exec_globals.values() if isinstance(v, cirq.Circuit)),
        None,
    )
    if llm_circuit is None:
        report["verdict"] = "no_circuit_found"
        return report

    # Step 3: Structure
    structure = check_structure(
        llm_circuit,
        control_circuit,
        original_prompt=original_prompt,
        llm_code=llm_code,
    )
    report["structure"] = structure

    # Step 4: Semantics
    semantics = check_semantics(llm_circuit, control_circuit)
    report["semantics"] = semantics

    # Step 5: Final Outcome
    if not structure["qubit_count_match"]:
        report["verdict"] = "missing_qubits"
    elif not structure["gate_types_match"]:
        report["verdict"] = "wrong_gate"
    elif not structure["measurement_match"]:
        report["verdict"] = "missing_measurement"
    elif not structure["simulation_match"]:
        report["verdict"] = "simulation_fail"
    elif not semantics["passed"]:
        report["verdict"] = "semantic_fail"
    else:
        report["verdict"] = "success"

    return report

In [17]:
code = """python\nimport cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create the circuit\ncircuit = cirq.Circuit(\n    # Apply Hadamard to qubit0\n    cirq.H(qubit0),\n    # Apply CNOT with qubit0 as control and qubit1 as target\n    cirq.CNOT(qubit0, qubit1)\n)\n\n# Print the circuit\nprint("Circuit to prepare |⟩:")\nprint(circuit)\n"""

if code.startswith("```python"):
    code = code.removeprefix("```python").removesuffix("```").strip()
elif code.startswith("python\n"):
    code = code.removeprefix("python\n")
exec(code)
 

Circuit to prepare |⟩:
0: ───H───@───
          │
1: ───────X───


In [18]:
q0,q1 = cirq.LineQubit.range(2)

bell_circuit = cirq.Circuit()
bell_circuit += cirq.H(q0)
bell_circuit += cirq.CNOT(q0,q1)
print(bell_circuit)



0: ───H───@───
          │
1: ───────X───


In [43]:
validate_circuit(code, bell_circuit, "entangle two qubits")

Circuit to prepare |⟩:
0: ───H───@───
          │
1: ───────X───
Circuit to prepare |⟩:
0: ───H───@───
          │
1: ───────X───


{'syntax': {'passed': True, 'error': None},
 'structure': {'qubit_count_match': True,
  'gate_types_match': True,
  'llm_gates': ['CXPowGate', 'HPowGate'],
  'control_gates': ['CXPowGate', 'HPowGate'],
  'measurement_match': True,
  'simulation_requested': False,
  'code_contains_simulation': False,
  'simulation_match': True,
  'circuit_depth_llm': 2,
  'circuit_depth_control': 2,
  'depth_within_tolerance': True},
 'semantics': {'passed': True,
  'fidelity_score': 0.9999998807907104,
  'distribution_match': True,
  'llm_distribution': {'(3,)': 0.453, '(0,)': 0.547},
  'control_distribution': {'(3,)': 0.513, '(0,)': 0.487}},
 'verdict': 'success'}

In [ ]:
import json
import cirq
from huggingface_hub import InferenceClient

def call_llm(prompt: str) -> str:
        client = InferenceClient(api_key="")
        response = client.text_generation(
        prompt=prompt,
        model="meta-llama/Llama-3.1-8B-Instruct", 
        max_new_tokens=1000
    )
        
def build_repair_prompt(original_prompt, bad_code, validation_result):
    # Mapping deterministic error categories to clear, non-negotiable feedback
    ERROR_MESSAGES = {
        "unrequested_measurement": "- Do NOT include measurement gates. The prompt requested only a circuit.",
        "unrequested_simulation": "- Do NOT include a simulator or run/simulate commands.",
        "missing_expected_gate": f"- Missing required gates. Your circuit must include: {validation_result['expected']['expected_gates']}.",
        "invalid_code": "- The syntax is invalid or could not be parsed by Cirq.",
        "simulation_fail": "Do NOT include simulation attributes. The prompt reequested only a circuit."
    }
    
    # Format instructions based on exactly what failed
    dynamic_rules = []
    for error in validation_result["error_categories"]:
        if error in ERROR_MESSAGES:
            dynamic_rules.append(ERROR_MESSAGES[error])

    # Construct the next prompt payload
    repair_prompt = f"""
    The previous code you generated failed validation. 

    Original user request:
    {original_prompt}

    Failure:
    {validation_result['verdict']}
    
    Reason for failure:
    {validation_result['reason']}

    Bad code attempt:
    ```python
    {bad_code}``` """

        
def run_repair_loop(original_prompt, control_circuit = cirq.Circuit,  max_attempts=10):
    # Step 1: Initial attempt using your Strategy 4 (Structured JSON Prompt)
    current_prompt = original_prompt
    
    for attempt in range(max_attempts):
        print(f"--- Attempt {attempt + 1} of {max_attempts} ---")
   
    raw_output = call_llm(current_prompt)
 
    # Step 2: Extract the circuit and run it through the validator
    validation_result = validate_circuit(raw_output, control_circuit)
        
  # Step 3: Check if it passed all semantic/structural checks
    if validation_result["passed"]:
        print("Validation passed successfully!")
        return raw_output
         
    # Step 4: If it failed, generate a new prompt using the template facts
    else:
        bad_code = raw_output
        print(f"Validation failed. Errors: {validation_result['error_categories']}")
        current_prompt = build_repair_prompt(original_prompt, bad_code, validation_result)
        
# If it exhausts all 10 attempts without passing
raise RuntimeError("Model failed to generate a valid circuit within 10 attempts.")